In [11]:
from pathlib import Path
import sys
sys.executable
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr

REPO_ROOT = Path.cwd().parents[1]  # falls Notebook in notebooks/... liegt
DATA_PATH = REPO_ROOT / "data" / "all_heuristics_dataset.pkl"

OUT_DIR = REPO_ROOT / "data" / "outputs" / "ppp_global"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PPP_TEMPLATE_PATH = REPO_ROOT / "src" / "prompt_templates" / "ppp_with_refs.md"

print("REPO_ROOT:", REPO_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists(), DATA_PATH)
print("OUT_DIR:", OUT_DIR)
print("PPP_TEMPLATE_PATH exists:", PPP_TEMPLATE_PATH.exists(), PPP_TEMPLATE_PATH)

# make repo importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("sys.path[0]:", sys.path[0])


REPO_ROOT: /Users/emirhangunes/VSCode/ppp-performance-prediction
DATA_PATH exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/data/all_heuristics_dataset.pkl
OUT_DIR: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/ppp_global
PPP_TEMPLATE_PATH exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/src/prompt_templates/ppp_with_refs.md
sys.path[0]: /Users/emirhangunes/VSCode/ppp-performance-prediction


In [14]:
import os
from src.api.deepseek_config import load_deepseek_config
from src.api.deepseek_client import DeepSeekLLMClient


cfg = load_deepseek_config()
client = DeepSeekLLMClient(cfg)

# Preflight (Auth + basic response)
resp = client.generate('Return JSON only: {"ping":"pong"}', temperature=0.0, max_tokens=30)
print(resp.text[:120])


```json
{"ping":"pong"}
```


In [15]:
df = pd.read_pickle(DATA_PATH)

required = {"heuristic_id", "raw_app_type", "code", "objective"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["code", "objective"]).copy()
df["heuristic_id"] = df["heuristic_id"].astype(str)
df["raw_app_type"] = df["raw_app_type"].astype(str)
df["objective"] = pd.to_numeric(df["objective"], errors="coerce")
df = df.dropna(subset=["objective"]).reset_index(drop=True)

print("Usable rows:", len(df))
df.head(3)


Usable rows: 15507


,heuristic_id,raw_app_type,instance_scale,filename,strategy,algorithm,code,objective,task_name,parse_ok,is_timeout
0,pop_0_op_e1_n0_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n0_251224_134701.json,e1,The new algorithm assigns scores based on a co...,"import numpy as np\n\ndef score(item, bins):\n...",1.51534,BinPacking,True,False
1,pop_0_op_e1_n10_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n10_251224_134701.json,e1,The new algorithm calculates scores for each b...,"import numpy as np\n\ndef score(item, bins):\n...",0.32770,BinPacking,True,False
2,pop_0_op_e1_n11_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n11_251224_134701.json,e1,The new algorithm calculates scores for each b...,"import numpy as np\n\ndef score(item, bins):\n...",1.51534,BinPacking,True,False


In [16]:
task_bounds = (
    df.groupby("raw_app_type")["objective"]
      .agg(["min", "max"])
      .to_dict(orient="index")
)
task_bounds


{'bin_greedy': {'min': 0.00563, 'max': 1.51534},
 'cvrp_lns': {'min': 3343425.3, 'max': 3981765.6},
 'premarshalling_astar': {'min': 0.08148, 'max': 13.9329},
 'puzzle_astar': {'min': 0.4574, 'max': 3.65888}}

In [17]:
def select_refs_stratified(df_task: pd.DataFrame, target_id: str, k: int = 5):
    df_cand = df_task[df_task["heuristic_id"] != target_id].sort_values("objective", ascending=True)
    n = len(df_cand)
    if n < k:
        return None

    if k == 3:
        idxs = [0, n // 2, n - 1]  # best, mid, worst
    elif k == 5:
        idxs = [0, n // 4, n // 2, (3 * n) // 4, n - 1]  # best, q25, mid, q75, worst
    else:
        # fallback: evenly spaced k points (includes ends)
        idxs = np.linspace(0, n - 1, k).round().astype(int).tolist()

    # unique + keep order
    seen = set()
    refs = []
    for i in idxs:
        row = df_cand.iloc[int(i)]
        hid = str(row["heuristic_id"])
        if hid not in seen:
            refs.append(row)
            seen.add(hid)

    if len(refs) < k:
        # fill from the start if duplicates collapsed (rare but possible)
        for _, row in df_cand.iterrows():
            hid = str(row["heuristic_id"])
            if hid not in seen:
                refs.append(row)
                seen.add(hid)
            if len(refs) >= k:
                break

    return refs[:k]


def build_refs_block(refs):
    lines = []
    for i, r in enumerate(refs, 1):
        lines += [
            f"{i})",
            "Heuristic code:",
            "```python",
            str(r["code"]),
            "```",
            f"Objective: {float(r['objective'])}",
            ""
        ]
    return "\n".join(lines).strip() + "\n"


In [18]:
ppp_template = PPP_TEMPLATE_PATH.read_text(encoding="utf-8")

def render_ppp_prompt(*, raw_app_type, task_min, task_max, references_block, target_code):
    return ppp_template.format(
        raw_app_type=raw_app_type,
        task_min=task_min,
        task_max=task_max,
        references_block=references_block,
        target_code=target_code,
    )

print("Template loaded, length:", len(ppp_template))


Template loaded, length: 715


In [19]:
from src.api.parse import parse_ppp_response


In [20]:
TEST_PER_TASK = 3

for app, df_task in df.groupby("raw_app_type"):
    task_min = task_bounds[app]["min"]
    task_max = task_bounds[app]["max"]

    print("\n=== APP:", app, "| rows:", len(df_task), "| range:", (task_min, task_max))
    df_small = df_task.head(TEST_PER_TASK)

    for _, row in df_small.iterrows():
        hid = row["heuristic_id"]
        refs = select_refs_stratified(df_task, hid, k=5)
        if refs is None:
            print("No refs for", hid)
            continue

        prompt = render_ppp_prompt(
            raw_app_type=app,
            task_min=task_min,
            task_max=task_max,
            references_block=build_refs_block(refs),
            target_code=str(row["code"]),
        )

        resp = client.generate(prompt, temperature=0.2, max_tokens=512)
        pred, conf, ok, err, _ = parse_ppp_response(resp.text)

        if pred is not None:
            pred = max(task_min, min(task_max, pred))  # clamp to bounds

        print("hid:", hid, "| obj:", row["objective"], "| pred:", pred, "| conf:", conf, "| ok:", ok, "| err:", err)



=== APP: bin_greedy | rows: 4445 | range: (0.00563, 1.51534)
hid: pop_0_op_e1_n0_251224_134701 | obj: 1.51534 | pred: 0.089 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n10_251224_134701 | obj: 0.3277 | pred: 0.085 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n11_251224_134701 | obj: 1.51534 | pred: 0.045 | conf: 0.65 | ok: True | err: None

=== APP: cvrp_lns | rows: 1557 | range: (3343425.3, 3981765.6)
hid: pop_0_op_e1_n0_251225_153652 | obj: 3978416.0 | pred: 3785000.0 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n11_251225_153652 | obj: 3613762.5 | pred: 3620000.0 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n12_251225_153652 | obj: 3956702.6 | pred: 3620000.0 | conf: 0.65 | ok: True | err: None

=== APP: premarshalling_astar | rows: 4647 | range: (0.08148, 13.9329)
hid: pop_0_op_e1_n0_250815_114538 | obj: 10.69957 | pred: 0.452 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n10_250815_114538 | obj: 8.79957 | pred: 0.45 | conf: 0.65 | ok: True 

In [ ]:
PPP_PKL = OUT_DIR / "ppp_results_code.pkl"
PPP_CSV = OUT_DIR / "ppp_results_code.csv"

if PPP_PKL.exists():
    prev = pd.read_pickle(PPP_PKL)
    prev["heuristic_id"] = prev["heuristic_id"].astype(str)
    done_ids = set(prev["heuristic_id"])
    results = prev.to_dict(orient="records")
    print("Resuming, loaded:", len(done_ids))
else:
    done_ids = set()
    results = []

t0 = time.time()

for app, df_task in df.groupby("raw_app_type"):
    task_min = task_bounds[app]["min"]
    task_max = task_bounds[app]["max"]

    for _, row in tqdm(df_task.iterrows(), total=len(df_task), desc=f"PPP [{app}]"):
        hid = str(row["heuristic_id"])
        if hid in done_ids:
            continue

        refs = select_refs_stratified(df_task, hid, k=5)
        if refs is None:
            continue

        prompt = render_ppp_prompt(
            raw_app_type=app,
            task_min=task_min,
            task_max=task_max,
            references_block=build_refs_block(refs),
            target_code=str(row["code"]),
        )

        resp = client.generate(prompt, temperature=0.2, max_tokens=512, meta={"stage":"ppp","heuristic_id":hid})
        pred, conf, ok, err, _ = parse_ppp_response(resp.text)

        if pred is not None:
            pred = max(task_min, min(task_max, pred))

        results.append({
            "heuristic_id": hid,
            "raw_app_type": app,
            "objective": float(row["objective"]),
            "prediction": pred,
            "confidence": conf,
            "parse_ok": bool(ok),
            "parse_error": err,
        })
        done_ids.add(hid)

        if len(results) % 200 == 0:
            tmp = pd.DataFrame(results)
            tmp.to_pickle(PPP_PKL)
            tmp.to_csv(PPP_CSV, index=False)
            print("[checkpoint] saved", len(results))

elapsed = time.time() - t0
ppp_df = pd.DataFrame(results)
ppp_df.to_pickle(PPP_PKL)
ppp_df.to_csv(PPP_CSV, index=False)

print("DONE. Rows:", len(ppp_df), "Time(s):", round(elapsed, 1))
print("Saved:", PPP_PKL)
print("Saved:", PPP_CSV)
ppp_df.head()
